# Train Main

Run a single training job and plot MCC/F1/AUPRC over epochs using saved metrics.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "training").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, matthews_corrcoef, average_precision_score
import torch

from data import ESMCSingleDS
from models import SequenceActiveSiteHead
from training import EPTrainer, SinglePipeline


/Users/connorott/PycharmProjects/bioml/.venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [3]:
data_name = "IEDB_Jespersen"
model_name = "esmc_300m"
base_data_dir = PROJECT_ROOT / "data" / "data_files"
save_dir = PROJECT_ROOT / "experiments" / "single_run_example"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
)


In [4]:
dataset = ESMCSingleDS(data_name, model_name, save_dir=base_data_dir)


Dropped 4 sequences over len 5000


In [6]:
params = {
    "hidden_dim": 256,
    "dropout": 0.1,
    "activation": "relu",
    "layers": 4,
    "kernel_size": 5,
    "block_type": "Conv1dInvBottleNeck",
    "lr": 1e-4,
    "weight_decay": 0.01,
    "max_tokens": 10000,
    "gamma": 2.0,
    "alpha": 0.5,
    "scheduler_type": "cosine",
}

pipeline = SinglePipeline(
    dataset,
    SequenceActiveSiteHead,
    EPTrainer,
    save_dir=save_dir,
    device=device,
    epochs=20,
)

best_score = pipeline.run(params)
print(f"Best validation AUPRC: {best_score:.4f}")


Epochs:   0%|          | 0/20 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
def load_metrics(path):
    data = np.load(path)
    return {k: data[k] for k in data.files}

train_metrics = load_metrics(save_dir / "data" / "train_metrics.npz")
val_metrics = load_metrics(save_dir / "data" / "val_metrics.npz")


In [ ]:
def sigmoid(x):
    x = np.clip(x, -50, 50)
    return 1 / (1 + np.exp(-x))


def compute_epoch_metrics(metrics):
    labels = metrics["labels"]
    logits = metrics["logits"]

    mcc = []
    f1 = []
    auprc = []

    for epoch in range(len(logits)):
        probs = sigmoid(logits[epoch])
        preds = (probs > 0.5).astype(int)
        mcc.append(matthews_corrcoef(labels[epoch], preds))
        f1.append(f1_score(labels[epoch], preds, zero_division=0))
        auprc.append(average_precision_score(labels[epoch], probs))

    return {
        "mcc": np.array(mcc),
        "f1": np.array(f1),
        "auprc": np.array(auprc),
    }

train_scores = compute_epoch_metrics(train_metrics)
val_scores = compute_epoch_metrics(val_metrics)


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(train_scores["mcc"], label="Train")
axes[0].plot(val_scores["mcc"], label="Val")
axes[0].set_ylabel("MCC")
axes[0].legend()

axes[1].plot(train_scores["f1"], label="Train")
axes[1].plot(val_scores["f1"], label="Val")
axes[1].set_ylabel("F1")
axes[1].legend()

axes[2].plot(train_scores["auprc"], label="Train")
axes[2].plot(val_scores["auprc"], label="Val")
axes[2].set_ylabel("AUPRC")
axes[2].set_xlabel("Epoch")
axes[2].legend()

fig.suptitle("Train vs Val Metrics Over Epochs")
plt.tight_layout()
